In [ ]:
import os
# set cwd to the test directory


In [ ]:

import sys
from pathlib import Path
root = Path(os.environ.get("PROTOSLEEPNET_DATA","data"))

sys.path.insert(0, str(root ))

sys.path.insert(0, str(root / "data" ))
sys.path.insert(0, str(root / "models" ))
sys.path.insert(0, str(root / "train" ))
sys.path.insert(0, str(root / "explain" ))

print( sys.path )

In [ ]:
# set cwd to the test directory
import torch
import os 

from data.dataset import get_dataloaders, PhysioExDataset
from models.protosleepnet import ProtoSleepNet, ProtoSleepNetTrainer
import torch

    
dataset = PhysioExDataset(
    datasets = [ "mass" ]
)
_, loader, _ = get_dataloaders( dataset )

path = [ p for p in os.listdir( "checkpoints/train_0/") if p.startswith("epoch=") and p.endswith(".pt") ][0]

model = ProtoSleepNet(
    in_chan = dataset.get_num_channels(),
    cdropout = 1.0,
    cmnlayers = 4,
)
model, _, _= ProtoSleepNetTrainer.load_checkpoint( model, path=os.path.join( "checkpoints/train_0/", path ) )
model = model.eval()


In [ ]:
from explain.prototypes.local import get_prototypes, PrototypeRelevance, proj_fn
from explain.prototypes.reconstruct import data_driven_reconstruction, model_driven_reconstructions

stages = [ "W", "N1", "N2", "N3", "R" ]
prototypes = get_prototypes( model )

with torch.no_grad():
    proto_preds = model.clf( prototypes ).cpu()
    proto_preds = torch.nn.functional.softmax( proto_preds, dim=-1 )

# for each prototype print the stages with confidence higher than 0.2 on one line
for i, proto_pred in enumerate( proto_preds ):
    labels = [ f"{stage}: {conf:.3f}" for stage, conf in zip( stages, proto_pred ) if conf > 0.2 ]
    print( f"Prototype {i}: " + ", ".join( labels ) )

In [ ]:
index = 14
n = 1
device = torch.device("cuda:1")

data_recon = data_driven_reconstruction(
    loader,
    model,
    index=index,
    device=device,
    n=n,
)

data_recon = data_recon.unsqueeze(0) if n == 1 else data_recon

# compute the similarity between the reconstruction and the prototypes
print( "Similarity between data reconstruction and prototypes:" )
with torch.no_grad():
    data_proj = proj_fn( model.to(device), data_recon.to( device ) ).cpu() 
    prototype = get_prototypes( model, index = index )

sim = torch.nn.functional.cosine_similarity(
    data_proj,
    prototype.unsqueeze(0),
    dim=-1,
)

print( sim.mean() )

n = len( data_recon ) if data_recon is not None else 0

# optimize the prototypes to reconstruct the data point
hybrid_recons = model_driven_reconstructions(
    model,
    index = index,
    device = device,
    n = n,
    init = data_recon,
    steps = 1000,
    lr = 1e-3,
)

print( "Similarity between hybrid reconstruction and prototypes:")
hybrid_recons = hybrid_recons.unsqueeze(0) if n == 1 else hybrid_recons 

with torch.no_grad():
    hybrid_proj = proj_fn( model.to(device), hybrid_recons.to(device) ).cpu()
    
sim = torch.nn.functional.cosine_similarity(
    hybrid_proj,
    prototype.unsqueeze(0),
    dim=-1,
)

print( sim.mean() )

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-darkgrid")

import numpy as np


In [ ]:
# now let's compute the relevance for this instance
hybrid_recons = hybrid_recons.reshape( n, 3, 29, 129 )

explainer = PrototypeRelevance(
    model = model.to(device),
    index = index,
    device = device,
    steps = 100,
    expects_batch=True
).to( device )



In [ ]:
CHANNELS = [ "EEG", "EOG", "EMG"]
fs = 100
freqs = np.arange( 0, 129 ) * fs / 2 / 128
times = np.arange( 0, 29 ) 

In [ ]:
mean, std = dataset.get_scaling()

def _psd_mean_over_time(x):
    # Accept shapes: (n, C, T, F) or (C, T, F)
    if x is None:
        return None
    if x.dim() == 3:
        x = x.unsqueeze(0)
    x = x.detach().cpu()
    
    x = x * std + mean
    
    psd = x.mean(dim=0).mean(dim=1)  # (C, F), mean over n and T
    return psd

rel = explainer( hybrid_recons.to(device), steps = 200 ) * std.to( device )

mean_instance = torch.zeros_like( hybrid_recons )
psd_min = _psd_mean_over_time(hybrid_recons)
psd_mean = _psd_mean_over_time(mean_instance)
psd_rel = torch.nn.functional.relu(rel.detach().cpu())
psd_rel = psd_rel.mean(dim=0).mean(dim=1)  # (C, F), mean over n and T
    
num_chans = min(3, psd_min.shape[0], psd_rel.shape[0], psd_mean.shape[0])

fig, axes = plt.subplots(2, num_chans, figsize=(6, 5), sharex=True, sharey='row')
if num_chans == 1:
    axes = np.array([[axes[0]], [axes[1]]])

for ch in range(num_chans):
    axes[0, ch].plot(freqs, psd_mean[ch].numpy(), label="Mean", linestyle= "--")
    axes[0, ch].plot(freqs, psd_min[ch].numpy(), label="Signal")
    axes[0, ch].set_title(f"{CHANNELS[ch]}")
    axes[0, ch].grid(True, alpha=0.3)
    axes[1, ch].plot(freqs, psd_rel[ch].numpy(), label="relevance (smoothed)")
    axes[1, ch].grid(True, alpha=0.3)
    axes[1, ch].set_xlabel("Frequency (Hz)")

axes[0, 0].set_ylabel("PSD")
axes[1, 0].set_ylabel("Relevance")

axes[0, -1].legend()

plt.tight_layout()
plt.savefig("prototype_relevance.pdf", bbox_inches="tight")